In [1]:
import time
import numpy as np
import pandas as pd
from pathlib import Path
import joblib

from sklearn.base import clone
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_recall_curve,
                             precision_score, recall_score, f1_score, confusion_matrix)
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
PROJECT_ROOT = Path(".").resolve()
DATA_FILE = PROJECT_ROOT / "7-days-scraping_preprocessed.csv"
OUT_DIR = PROJECT_ROOT / "models"
OUT_DIR.mkdir(exist_ok=True)

data = pd.read_csv(DATA_FILE, parse_dates=["submit_time"])
META = ["id", "title", "submit_time", "viral", "split"]
FEATURES = [c for c in data.columns if c not in META]

train = data[data["split"] == "train"]
test = data[data["split"] == "test"]
X_train, y_train = train[FEATURES], train["viral"]
X_test, y_test = test[FEATURES], test["viral"]

print(f"Features: {len(FEATURES)}")
print(f"Train: {len(train):,} posts, {y_train.sum()} viral ({y_train.mean():.1%})  "
      f"[{train['submit_time'].min():%d %b} -> {train['submit_time'].max():%d %b %H:%M}]")
print(f"Test:  {len(test):,} posts, {y_test.sum()} viral ({y_test.mean():.1%})  "
      f"[{test['submit_time'].min():%d %b %H:%M} -> {test['submit_time'].max():%d %b %H:%M}]")

Features: 25
Train: 5,388 posts, 164 viral (3.0%)  [07 Nov -> 12 Nov 17:56]
Test:  1,348 posts, 58 viral (4.3%)  [12 Nov 17:56 -> 13 Nov 22:18]


## 1. Candidate models



In [2]:
candidates = {
    "Logistic regression": [
        make_pipeline(StandardScaler(), LogisticRegression(C=C, class_weight="balanced", max_iter=5000))
        for C in [0.01, 0.1, 1.0]
    ],
    "Random forest": [
        RandomForestClassifier(n_estimators=400, min_samples_leaf=leaf, max_features="sqrt",
                               class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE)
        for leaf in [3, 10]
    ],
    "Gradient boosting": [
        HistGradientBoostingClassifier(learning_rate=lr, max_leaf_nodes=leaves, max_iter=300,
                                       l2_regularization=1.0, class_weight="balanced",
                                       early_stopping=False, random_state=RANDOM_STATE)
        for lr, leaves in [(0.05, 15), (0.03, 7)]
    ],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rows, best = [], {}
for name, models in candidates.items():
    for m in models:
        t0 = time.time()
        oof = cross_val_predict(m, X_train, y_train, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]
        ap, auc = average_precision_score(y_train, oof), roc_auc_score(y_train, oof)
        params = {k: v for k, v in m.get_params().items()
                  if k.split("__")[-1] in ("C", "min_samples_leaf", "learning_rate", "max_leaf_nodes")}
        rows.append({"model": name, "params": {k.split("__")[-1]: v for k, v in params.items()},
                     "CV PR-AUC": ap, "CV ROC-AUC": auc, "sec": time.time() - t0})
        if name not in best or ap > best[name]["ap"]:
            best[name] = {"model": m, "ap": ap, "oof": oof}

cv_results = pd.DataFrame(rows).sort_values("CV PR-AUC", ascending=False)
cv_results.round(3)

,model,params,CV PR-AUC,CV ROC-AUC,sec
3,Random forest,"{'max_leaf_nodes': None, 'min_samples_leaf': 3}",0.722,0.940,2.669
4,Random forest,"{'max_leaf_nodes': None, 'min_samples_leaf': 10}",0.717,0.945,0.820
5,Gradient boosting,"{'learning_rate': 0.05, 'max_leaf_nodes': 15, ...",0.690,0.930,0.468
0,Logistic regression,{'C': 0.01},0.686,0.940,2.767
6,Gradient boosting,"{'learning_rate': 0.03, 'max_leaf_nodes': 7, '...",0.686,0.939,0.372
1,Logistic regression,{'C': 0.1},0.676,0.941,1.937
2,Logistic regression,{'C': 1.0},0.667,0.940,1.961


## 2. Choose decision thresholds (train only)



In [3]:
def best_f1_threshold(y, p):
    prec, rec, thr = precision_recall_curve(y, p)
    f1 = 2 * prec * rec / np.clip(prec + rec, 1e-12, None)
    i = int(np.nanargmax(f1[:-1]))
    return float(thr[i])

for name, b in best.items():
    b["threshold"] = best_f1_threshold(y_train, b["oof"])
    print(f"{name:<20} CV PR-AUC {b['ap']:.3f}  ->  threshold {b['threshold']:.3f}")

Logistic regression  CV PR-AUC 0.686  ->  threshold 0.940
Random forest        CV PR-AUC 0.722  ->  threshold 0.599
Gradient boosting    CV PR-AUC 0.690  ->  threshold 0.677


## 3. Final evaluation on the test set



In [4]:
rng = np.random.default_rng(RANDOM_STATE)
boot_idx = [rng.integers(0, len(y_test), len(y_test)) for _ in range(1000)]
yt = y_test.to_numpy()

def ci(metric, p):
    vals = [metric(yt[i], p[i]) for i in boot_idx if yt[i].sum() > 0]
    return np.percentile(vals, [2.5, 97.5])

results, test_proba = [], {}
for name, b in best.items():
    model = clone(b["model"]).fit(X_train, y_train)
    p = model.predict_proba(X_test)[:, 1]
    pred = (p >= b["threshold"]).astype(int)
    b["fitted"], test_proba[name] = model, p
    lo, hi = ci(average_precision_score, p)
    results.append({"model": name,
                    "PR-AUC": average_precision_score(yt, p), "PR-AUC 95% CI": f"{lo:.2f}–{hi:.2f}",
                    "ROC-AUC": roc_auc_score(yt, p),
                    "precision": precision_score(yt, pred, zero_division=0),
                    "recall": recall_score(yt, pred), "F1": f1_score(yt, pred),
                    "flagged": int(pred.sum())})

rule = test["on_front_page_by_N"].to_numpy().astype(int)
results.append({"model": "Rule: on front page by 60 min",
                "PR-AUC": average_precision_score(yt, rule), "PR-AUC 95% CI": "",
                "ROC-AUC": roc_auc_score(yt, rule),
                "precision": precision_score(yt, rule), "recall": recall_score(yt, rule),
                "F1": f1_score(yt, rule), "flagged": int(rule.sum())})

test_results = pd.DataFrame(results).set_index("model")
print(f"Test set: {len(yt)} posts, {yt.sum()} viral | random-model PR-AUC ≈ {yt.mean():.3f}")
test_results.round(3)

Test set: 1348 posts, 58 viral | random-model PR-AUC ≈ 0.043


,PR-AUC,PR-AUC 95% CI,ROC-AUC,precision,recall,F1,flagged
model,,,,,,,
Logistic regression,0.712,0.58–0.82,0.958,0.696,0.672,0.684,56
Random forest,0.756,0.64–0.85,0.942,0.695,0.707,0.701,59
Gradient boosting,0.738,0.62–0.85,0.953,0.672,0.672,0.672,58
Rule: on front page by 60 min,0.321,,0.904,0.359,0.879,0.510,142


## 4. The chosen model in detail

The final model is the one with the best **cross-validated** PR-AUC on train (not the best test score — picking by test score would overfit the test set).

In [5]:
final_name = max(best, key=lambda n: best[n]["ap"])
final = best[final_name]
p_final = test_proba[final_name]
pred_final = (p_final >= final["threshold"]).astype(int)

tn, fp, fn, tp = confusion_matrix(yt, pred_final).ravel()
print(f"Final model: {final_name}  (threshold {final['threshold']:.3f})\n")
print("Confusion matrix on test:")
print(pd.DataFrame([[tn, fp], [fn, tp]],
                   index=["actual: not viral", "actual: viral"],
                   columns=["predicted: not viral", "predicted: viral"]).to_string())
print(f"\n-> catches {tp} of {tp + fn} viral posts ({tp / (tp + fn):.0%} recall)")
print(f"-> of {tp + fp} posts it flags, {tp} really go viral ({tp / max(tp + fp, 1):.0%} precision)")

Final model: Random forest  (threshold 0.599)

Confusion matrix on test:
                   predicted: not viral  predicted: viral
actual: not viral                  1272                18
actual: viral                        17                41

-> catches 41 of 58 viral posts (71% recall)
-> of 59 posts it flags, 41 really go viral (69% precision)


### Precision / recall trade-off

The threshold can be moved depending on the use case: lower it to catch more viral posts (more false alarms), raise it for fewer, surer picks.

In [6]:
rows = []
for target_recall in [0.9, 0.8, 0.7, 0.6, 0.5]:
    prec, rec, thr = precision_recall_curve(y_train, final["oof"])
    ok = np.where(rec[:-1] >= target_recall)[0]
    t = float(thr[ok[-1]])                  # highest threshold that still reaches the target recall on train
    pr = (p_final >= t).astype(int)
    rows.append({"target recall (train)": target_recall, "threshold": round(t, 3),
                 "test recall": recall_score(yt, pr), "test precision": precision_score(yt, pr, zero_division=0),
                 "posts flagged": int(pr.sum())})
pd.DataFrame(rows).round(3)

,target recall (train),threshold,test recall,test precision,posts flagged
0,0.9,0.033,0.931,0.148,364
1,0.8,0.111,0.879,0.338,151
2,0.7,0.665,0.707,0.707,58
3,0.6,0.781,0.638,0.787,47
4,0.5,0.855,0.586,0.810,42


## 5. What drives the predictions?

Permutation importance on the test set: how much test PR-AUC drops when one feature's values are shuffled. Bigger drop = the model relies on it more.

In [7]:
imp = permutation_importance(final["fitted"], X_test, y_test, scoring="average_precision",
                             n_repeats=20, random_state=RANDOM_STATE, n_jobs=-1)
importance = (pd.DataFrame({"feature": FEATURES, "PR-AUC drop": imp.importances_mean, "std": imp.importances_std})
              .sort_values("PR-AUC drop", ascending=False).set_index("feature"))
importance.head(12).round(4)

,PR-AUC drop,std
feature,,
best_top_rank,0.1808,0.0456
score_per_min,0.0855,0.0338
score_gain_2nd_half,0.0720,0.0395
score_at_N,0.0562,0.0331
comments_per_point,0.0313,0.0112
on_front_page_by_N,0.0244,0.0295
comments_per_min,0.0241,0.0104
comments_at_N,0.0165,0.0063
title_len,0.0120,0.0137


## 6. Examples: hits and misses

In [8]:
show = test[["id", "title", "viral"]].assign(probability=p_final, predicted=pred_final)
show["score_at_60min"] = np.expm1(test["score_at_N"]).round().astype(int)   # undo log1p for readability
show["front_page_rank"] = test["best_top_rank"].astype(int).replace(501, -1)  # -1 = never on front page
cols = ["title", "score_at_60min", "front_page_rank", "probability"]
pd.set_option("display.max_colwidth", 60)

print("Top-confidence correct calls (true positives):")
print(show[(show.viral == 1) & (show.predicted == 1)].nlargest(5, "probability")[cols].round(3).to_string(index=False))
print("\nMissed viral posts (false negatives):")
print(show[(show.viral == 1) & (show.predicted == 0)].nlargest(5, "probability")[cols].round(3).to_string(index=False))
print("\nFalse alarms (false positives):")
print(show[(show.viral == 0) & (show.predicted == 1)].nlargest(5, "probability")[cols].round(3).to_string(index=False))

Top-confidence correct calls (true positives):
                                                                        title  score_at_60min  front_page_rank  probability
                            "LibreOffice works 365 days a year. Just saying."             266                1        0.996
    AI unlocks Gödel's notebooks, said to contain hundreds of proved theorems              66                4        0.990
                             "Doomsday: Friday, 13 November, A.D. 2026 (1960)              75                1        0.987
              The Netherlands Is Dismantling the International Criminal Court              95                1        0.987
Dwarf planet candidate appears which takes roughly 280 years to orbit our sun              42                3        0.986

Missed viral posts (false negatives):
                                                                        title  score_at_60min  front_page_rank  probability
          Found: A protein that drives ageing 

## 7. Save the model

In [9]:
joblib.dump({"model": final["fitted"], "threshold": final["threshold"], "features": FEATURES,
             "name": final_name}, OUT_DIR / "viral_classifier.joblib")
print("Saved:", OUT_DIR / "viral_classifier.joblib")

Saved: C:\Users\chahi\Desktop\Lab Networking project\Classification\models\viral_classifier.joblib
